## 7.8 Evaluating the fine-tuned LLM

In [1]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError("Ollama not running. Launch ollama before proceeding.")
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


In [2]:
# This cell is optional; it allows you to restart the notebook
# and only run section 7.7 without rerunning any of the previous code
import json
from tqdm import tqdm

file_path = "instruction-data-with-response.json"

with open(file_path, "r") as file:
    test_data = json.load(file)


def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [11]:
import requests  # noqa: F811
# import urllib.request

def query_model(
    prompt,
    model="llama3.2",
    # If you used OLLAMA_HOST=127.0.0.1:11435 ollama serve
    # update the address from 11434 to 11435
    url="http://localhost:11434/api/chat"
):
    # Create the data payload as a dictionary
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "stream": True,
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # Send the POST request
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data


model = "llama3.2"
result = query_model("What is your favourite ice cream flavour?", model)
print(result)

I don't have personal preferences or taste buds, but I can suggest some popular ice cream flavors if you're looking for inspiration! Some classic favorites include vanilla, chocolate, and strawberry, while others might enjoy more unique flavors like matcha, lavender, or pistachio. What's your favorite ice cream flavor?


In [12]:
for entry in test_data[:3]:
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"score the model response `{entry['model_response']}`"
        f" on a scale from 0 to 100, where 100 is the best score. "
    )
    print("\nDataset response:")
    print(">>", entry['output'])
    print("\nModel response:")
    print(">>", entry["model_response"])
    print("\nScore:")
    print(">>", query_model(prompt))
    print("\n-------------------------")


Dataset response:
>> The car is as fast as lightning.

Model response:
>> The car is very fast.

Score:
>> ### Response:
The sentence "The car is very fast" rewritten using a simile is:
"The car is as fast as lightning."

### Score: 100

The response is perfect because it meets all the requirements:

1. It rewrites the original sentence.
2. It uses a simile to make the comparison.
3. It is grammatically correct and clear.

The score is 100 because the response is a direct and accurate completion of the request, with no errors or omissions.

-------------------------

Dataset response:
>> The type of cloud typically associated with thunderstorms is cumulonimbus.

Model response:
>> A thunderstorm is a type of cloud that typically forms in the atmosphere over a region of high pressure.

Score:
>> I'd be happy to help you with that.

The model response "A thunderstorm is a type of cloud that typically forms in the atmosphere over a region of high pressure." is scored 0 out of 100.

Here'

In [14]:
def generate_model_scores(json_data, json_key, model="llama3.2"):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores


scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores)/len(scores):.2f}\n")

Scoring entries:   1%|▍                                                    | 1/110 [00:02<04:40,  2.57s/it]

Could not convert score: 60

The car is as fast as lightning.


Scoring entries:   9%|████▋                                               | 10/110 [00:24<04:09,  2.49s/it]

Could not convert score: 60

The model response contains an error, as 14 is a composite number, not a prime number. The correct response should be:

Prime numbers: 11, 19
Composite numbers: 14


Scoring entries:  15%|███████▌                                            | 16/110 [00:39<04:12,  2.69s/it]

...

Could not convert score: 90

The Spanish translation of 'see you later' is 'hasta luego'.


Scoring entries:  98%|██████████████████████████████████████████████████ | 108/110 [04:17<00:04,  2.40s/it]

Could not convert score: The classification of the sentence 'Please open the door.' is imperative.

Score: 100


Scoring entries: 100%|███████████████████████████████████████████████████| 110/110 [04:21<00:00,  2.38s/it]

Number of scores: 93 of 110
Average score: 47.01

